# Bluestock Mutual Fund Capstone — Day 4: Performance Analytics & Scoring
This notebook calculates advanced risk-adjusted performance statistics for all 40 schemes, including:
- **Daily Returns & Validation**: Computation and verification of returns.
- **CAGR Comparisons**: Annualized returns over 1-Year, 3-Year, and 5-Year (4.4-Year data limit) horizons.
- **Sharpe and Sortino Ratios**: Risk-adjusted returns (using daily RBI repo proxy of 6.5% risk-free rate).
- **Alpha and Beta Regression**: OLS regression against Nifty 100.
- **Maximum Drawdowns**: Minimum cumulative peak-to-trough returns and worst date ranges.
- **Fund Scorecard**: A composite scoring ranking system (0-100).

In [ ]:
import os
import sqlite3
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
db_path = '../data/db/bluestock_mf.db'
conn = sqlite3.connect(db_path)
print('Connected to SQLite Database successfully!')

## 1. Daily Returns Calculation & Distribution
Calculating daily percentage changes of fund NAVs on trading business days and plotting returns distributions.

In [ ]:
# Load daily NAVs excluding weekends
sql_nav = '''
SELECT n.nav_date, n.nav_value, n.amfi_code, f.scheme_name
FROM fact_nav n
JOIN dim_fund f ON n.amfi_code = f.amfi_code
JOIN dim_date d ON n.nav_date = d.date_id
WHERE d.is_weekend = 0
ORDER BY n.amfi_code, n.nav_date
'''
df_nav = pd.read_sql_query(sql_nav, conn)
df_nav['nav_date'] = pd.to_datetime(df_nav['nav_date'])

# Calculate daily returns for a sample fund
sample_code = 119551 # SBI Bluechip
sample_nav = df_nav[df_nav['amfi_code'] == sample_code].copy()
sample_nav['daily_return'] = sample_nav['nav_value'].pct_change()

# Plotting distribution
plt.figure(figsize=(10, 4))
sns.histplot(sample_nav['daily_return'].dropna(), bins=60, kde=True, color='indigo')
plt.title(f'Daily Returns Distribution — {sample_nav["scheme_name"].iloc[0]}', fontsize=12, fontweight='bold')
plt.xlabel('Daily Return')
plt.ylabel('Frequency')
plt.show()

# Distribution statistics
print(sample_nav['daily_return'].describe())

## 2. Compounding Annual Growth Rate (CAGR) Comparison
CAGR is calculated as: $CAGR = (NAV_{end} / NAV_{start}) ^ {(1/n)} - 1$. We compile 1-Year, 3-Year, and Inception (4.4-Year) comparison tables for all 40 funds. Note that the 5-Year CAGR is reported as `NaN` due to data limits.

In [ ]:
# Load scorecard and print top 10 funds based on 3-Year CAGR
df_scorecard = pd.read_csv('../fund_scorecard.csv')
print('Top 10 Funds by 3-Year CAGR:')
display(df_scorecard[['amfi_code', 'scheme_name', 'cagr_3yr']].head(10))

## 3. Sharpe & Sortino Ratios (Risk-Adjusted Performance)
- **Sharpe Ratio**: $(R_p - R_f) / Std(R_p) \times \sqrt{252}$ using risk-free rate $R_f = 6.5\%$.
- **Sortino Ratio**: Denominator uses downside standard deviation (focusing on negative return days only).

In [ ]:
print('Top 10 Funds by Sharpe Ratio:')
display(df_scorecard[['amfi_code', 'scheme_name', 'sharpe_ratio']].sort_values('sharpe_ratio', ascending=False).head(10))

## 4. Alpha & Beta Regression Analysis
Computing linear regression parameters using Nifty 100 as the market index proxy. 

In [ ]:
df_alpha_beta = pd.read_csv('../alpha_beta.csv')
print('Sample Alpha/Beta coefficients:')
display(df_alpha_beta.head(10))

## 5. Maximum Drawdown & Worst Date Ranges
Calculating the peak-to-trough drop: $Drawdown = NAV_t / Cummax(NAV) - 1$. 

In [ ]:
print('Top 5 funds with the smallest maximum drawdown (most defensive):')
display(df_scorecard[['amfi_code', 'scheme_name', 'max_drawdown']].sort_values('max_drawdown', ascending=False).head(5))

## 6. Composite Fund Scorecard & Rankings
Composite scorecard rank from 0-100: `30% × 3yr return rank + 25% × Sharpe rank + 20% × Alpha rank + 15% × expense ratio rank (inverse) + 10% × max DD rank (inverse)`.

In [ ]:
print('Overall Fund Leaderboard (Composite Score):')
display(df_scorecard[['amfi_code', 'scheme_name', 'composite_score']].head(10))

## 7. Benchmark Comparison Chart & Tracking Error
Visualizing Top 5 funds relative to Nifty 50 and Nifty 100 with annualized Tracking Error: $TE = Std(R_p - R_m) \times \sqrt{252}$.

In [ ]:
# Display the generated figure
from IPython.display import Image, display as ipy_display
fig_path = '../reports/figures/benchmark_comparison.png'
if os.path.exists(fig_path):
    ipy_display(Image(filename=fig_path))
else:
    print('Benchmark chart figure not found!')

# Close connection
conn.close()